# AE2 · Reconocimiento de alquileres de bicicletas por hora

**Proyecto:** variación de los alquileres por hora entre días laborables y no laborables en Capital Bikeshare (2011–2012).

**Etapa 1:** obtener un dataset de Hugging Face, reconocer su estructura y generar automáticamente un resumen en texto. Esta entrega prepara el diseño de un análisis descriptivo; las comparaciones detalladas se desarrollarán en una etapa posterior.

**Escenario hipotético:** un equipo de planeación de un servicio de bicicletas compartidas necesita conocer los patrones horarios observados para orientar la revisión de horarios operativos. No se afirma que exista un problema operativo comprobado.

**Pregunta:** ¿Cómo varía el número de alquileres de bicicletas por hora entre días laborables y no laborables en Capital Bikeshare durante 2011 y 2012?

Ejecuta las celdas en orden. Se requiere conexión a Internet en la primera descarga y el entorno de `requirements.txt`. El dataset es público y se obtiene sin token. No se guarda una copia de los datos en el repositorio; Hugging Face utiliza su caché local.

## 1. Preparación y trazabilidad

Se utiliza el identificador `t22000t/bike-sharing-tabular`, configuración `default` y split `train`. Se fija el commit del repositorio para reproducir exactamente la versión consultada el **22 de octubre de 2026**. Se carga el split completo, sin muestreo, filtrado ni eliminación de filas o columnas.

La ficha de Hugging Face corresponde a una versión tabular del dataset **Bike Sharing** de UCI, originado en registros reales de Capital Bikeshare en Washington D. C. durante 2011–2012; no se presenta como un conjunto sintético. La procedencia, la licencia y el diccionario se documentan con más detalle en el documento del proyecto.

In [1]:
from pathlib import Path
from io import StringIO
from importlib.metadata import version
import hashlib
import json
import platform

import pandas as pd
from datasets import load_dataset
from huggingface_hub import HfApi
from IPython.display import display

# Detecta la raíz desde el repositorio o desde AE2/notebooks.
candidatas = [Path.cwd(), *Path.cwd().parents]
RAIZ = next(
    (ruta for ruta in candidatas
     if (ruta / "requirements.txt").is_file() and (ruta / "AE2").is_dir()),
    None,
)
if RAIZ is None:
    raise FileNotFoundError("Abre Jupyter desde la raíz del proyecto o AE2/notebooks.")

DATASET_ID = "t22000t/bike-sharing-tabular"
REVISION = "6b645e0cedbc5208fcca11ac94544b088478bf61"
CONFIGURACION = "default"
SPLIT = "train"
FECHA_CONSULTA = "2026-10-22"
ENLACE_DATASET = f"https://huggingface.co/datasets/{DATASET_ID}"
RESULTADOS = RAIZ / "AE2" / "resultados"

versiones = {paquete: version(paquete) for paquete in (
    "datasets", "pandas", "pyarrow", "huggingface-hub", "ipykernel", "ipywidgets", "nbclient", "nbformat", "jupyterlab"
)}
print("Python:", platform.python_version())
display(pd.Series(versiones, name="versión").to_frame())

Python: 3.12.14


,versión
datasets,5.1.0
pandas,3.1.0
pyarrow,26.0.0
huggingface-hub,2.2.0
ipykernel,7.4.0
ipywidgets,8.1.9
nbclient,0.11.0
nbformat,5.11.1
jupyterlab,4.6.4


## 2. Obtención desde Hugging Face y carga en Pandas

`load_dataset` descarga o recupera de la caché el split fijado por su revisión. `to_pandas()` lo convierte en el DataFrame `df`. Las siguientes revisiones y el archivo de texto se calculan a partir de ese mismo DataFrame. La etiqueta `train` es el nombre del split publicado; aquí no se entrena un modelo.

In [2]:
dataset = load_dataset(
    DATASET_ID,
    name=CONFIGURACION,
    split=SPLIT,
    revision=REVISION,
    token=False,
)
df = dataset.to_pandas()
assert isinstance(df, pd.DataFrame)

print(dataset)
print("Muestra: no; se utiliza todo el split publicado.")
print("Dimensiones del DataFrame:", df.shape)

Dataset({
    features: ['instant', 'dteday', 'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed', 'casual', 'registered', 'cnt'],
    num_rows: 17379
})
Muestra: no; se utiliza todo el split publicado.
Dimensiones del DataFrame: (17379, 17)


## 3. Revisión básica del DataFrame

Cada fila contiene un **registro horario agregado** de alquileres del sistema, identificado por una fecha (`dteday`) y una hora (`hr`), junto con variables de calendario, clima y conteos de alquileres. Una fila no corresponde a una persona ni a un viaje individual.

Las variables centrales para la necesidad son `cnt` (total de alquileres de la hora), `hr` (hora del día) y `workingday` (1: día que no es fin de semana ni festivo; 0: resto). `dteday` delimita el periodo y permite revisar la cobertura temporal. `casual` y `registered` permiten verificar la composición del total, sin identificar usuarios únicos. Los códigos enteros de calendario representan categorías, aunque Pandas los almacene como números.

In [3]:
# Algunos registros; no modifica el DataFrame.
display(df.head(8))

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0000,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0000,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0000,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0000,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0000,0,1,1
5,6,2011-01-01,1,0,1,5,0,6,0,2,0.24,0.2576,0.75,0.0896,0,1,1
6,7,2011-01-01,1,0,1,6,0,6,0,1,0.22,0.2727,0.80,0.0000,2,0,2
7,8,2011-01-01,1,0,1,7,0,6,0,1,0.20,0.2576,0.86,0.0000,1,2,3


In [4]:
filas, columnas = df.shape
print(f"Filas: {filas:,}")
print(f"Columnas: {columnas}")
print("Nombres de las columnas:")
print(df.columns.tolist())

Filas: 17,379
Columnas: 17
Nombres de las columnas:
['instant', 'dteday', 'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed', 'casual', 'registered', 'cnt']


In [5]:
tipos_y_nulos = pd.DataFrame({
    "tipo_pandas": df.dtypes.astype(str),
    "valores_nulos": df.isna().sum(),
})
tipos_y_nulos.index.name = "variable"
display(tipos_y_nulos)

,tipo_pandas,valores_nulos
variable,,
instant,int64,0
dteday,str,0
season,int64,0
yr,int64,0
mnth,int64,0
hr,int64,0
holiday,int64,0
weekday,int64,0
workingday,int64,0


In [6]:
buffer_info = StringIO()
df.info(buf=buffer_info, memory_usage="deep", show_counts=True)
info_dataframe = buffer_info.getvalue()
print(info_dataframe)

<class 'pandas.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   instant     17379 non-null  int64  
 1   dteday      17379 non-null  str    
 2   season      17379 non-null  int64  
 3   yr          17379 non-null  int64  
 4   mnth        17379 non-null  int64  
 5   hr          17379 non-null  int64  
 6   holiday     17379 non-null  int64  
 7   weekday     17379 non-null  int64  
 8   workingday  17379 non-null  int64  
 9   weathersit  17379 non-null  int64  
 10  temp        17379 non-null  float64
 11  atemp       17379 non-null  float64
 12  hum         17379 non-null  float64
 13  windspeed   17379 non-null  float64
 14  casual      17379 non-null  int64  
 15  registered  17379 non-null  int64  
 16  cnt         17379 non-null  int64  
dtypes: float64(4), int64(12), str(1)
memory usage: 2.4 MB



## 4. Comprobaciones mínimas de cobertura y coherencia

Se convierte `dteday` en una **serie auxiliar** de fechas para reconocer el periodo, sin agregar columnas a `df` ni cambiar sus tipos. También se revisa la combinación fecha–hora y la identidad `cnt = casual + registered`.

La ausencia de valores nulos dentro de las filas no significa que exista una fila para cada hora del calendario. Por eso se compara la cobertura publicada con las 24 horas de cada fecha del periodo; las horas sin registro no se completan con cero ni se imputan. Esta revisión no establece la causa de las ausencias.

In [7]:
fechas = pd.to_datetime(df["dteday"], format="%Y-%m-%d", errors="coerce")
fecha_minima = fechas.min()
fecha_maxima = fechas.max()
dias_del_periodo = int((fecha_maxima - fecha_minima).days + 1)
horas_calendario = dias_del_periodo * 24
pares_fecha_hora = pd.DataFrame({"fecha": fechas, "hora": df["hr"]})
horas_observadas = int(pares_fecha_hora.drop_duplicates().shape[0])
horas_sin_registro = horas_calendario - horas_observadas
duplicados_fecha_hora = int(pares_fecha_hora.duplicated().sum())
conteos_inconsistentes = int((df["cnt"] != df["casual"] + df["registered"]).sum())

cobertura = {
    "fecha_inicial": fecha_minima.strftime("%Y-%m-%d"),
    "fecha_final": fecha_maxima.strftime("%Y-%m-%d"),
    "fechas_distintas": int(fechas.nunique()),
    "fechas_no_interpretables": int(fechas.isna().sum()),
    "dias_del_periodo": dias_del_periodo,
    "horas_calendario": horas_calendario,
    "horas_observadas_distintas": horas_observadas,
    "horas_sin_registro": horas_sin_registro,
    "duplicados_fecha_hora": duplicados_fecha_hora,
    "filas_con_cnt_distinto_de_casual_mas_registered": conteos_inconsistentes,
}
display(pd.Series(cobertura, name="resultado").to_frame())
print("Registros por año (control de cobertura):")
display(fechas.dt.year.value_counts().sort_index().rename("registros").to_frame())
print("Registros por tipo de día (1: laborable; 0: no laborable):")
display(df["workingday"].value_counts().sort_index().rename("registros").to_frame())

,resultado
fecha_inicial,2011-01-01
fecha_final,2012-12-31
fechas_distintas,731
fechas_no_interpretables,0
dias_del_periodo,731
horas_calendario,17544
horas_observadas_distintas,17379
horas_sin_registro,165
duplicados_fecha_hora,0
filas_con_cnt_distinto_de_casual_mas_registered,0


Registros por año (control de cobertura):


,registros
dteday,
2011,8645
2012,8734


Registros por tipo de día (1: laborable; 0: no laborable):


,registros
workingday,
0,5514
1,11865


## 5. Generación automática del resumen en UTF-8

Esta celda crea `AE2/resultados/` si hace falta y **sobrescribe** `resumen_dataframe.txt` en cada ejecución. Incluye identificador, revisión, configuración, split, uso del split completo, dimensiones, nombres y tipos de columnas y valores nulos calculados a partir de `df`. Se agrega la cobertura reconocida y una huella SHA-256 del contenido y del esquema del DataFrame para vincular los artefactos.

También se genera `ficha_tecnica.json` como apoyo verificable para el documento: contiene los mismos resultados, las versiones usadas y el tamaño de los archivos de datos según los metadatos de Hugging Face. Este JSON es un resultado auxiliar, no reemplaza el resumen solicitado.

In [8]:
# Metadatos de la misma revisión fijada; no descarga otra tabla.
info_hf = HfApi().dataset_info(
    repo_id=DATASET_ID,
    revision=REVISION,
    files_metadata=True,
    token=False,
)
assert info_hf.sha == REVISION
archivos_datos = [
    {"ruta": archivo.rfilename, "bytes": archivo.size}
    for archivo in info_hf.siblings
    if archivo.rfilename.endswith((".parquet", ".csv", ".jsonl", ".arrow"))
]

esquema_serializado = json.dumps(
    [(str(nombre), str(tipo)) for nombre, tipo in df.dtypes.items()],
    ensure_ascii=False,
).encode("utf-8")
huella = hashlib.sha256(
    pd.util.hash_pandas_object(df, index=True).to_numpy().tobytes() + esquema_serializado
).hexdigest()

lineas = [
    "RESUMEN DEL DATAFRAME — AE2",
    f"Dataset: {DATASET_ID}",
    f"Enlace: {ENLACE_DATASET}",
    f"Fecha de consulta: {FECHA_CONSULTA}",
    f"Revisión: {REVISION}",
    f"Configuración: {CONFIGURACION}",
    f"Split: {SPLIT}",
    "Muestra: no; split completo, sin filtros ni eliminación de registros.",
    f"Filas: {filas}",
    f"Columnas: {columnas}",
    "",
    "NOMBRES, TIPOS DE DATOS Y VALORES NULOS",
    "variable | tipo_pandas | valores_nulos",
]
for nombre in df.columns:
    lineas.append(f"{nombre} | {df[nombre].dtype} | {int(df[nombre].isna().sum())}")
lineas += [
    "",
    "COBERTURA Y COHERENCIA BÁSICA",
    f"Periodo: {cobertura['fecha_inicial']} a {cobertura['fecha_final']}",
    f"Fechas distintas: {cobertura['fechas_distintas']}",
    f"Horas de calendario en el periodo: {horas_calendario}",
    f"Horas observadas distintas: {horas_observadas}",
    f"Horas sin registro: {horas_sin_registro} (no imputadas; no equivalen a cero alquileres)",
    f"Duplicados fecha-hora: {duplicados_fecha_hora}",
    f"Filas con cnt distinto de casual + registered: {conteos_inconsistentes}",
    "",
    f"Huella SHA-256 del DataFrame y esquema: {huella}",
]
resumen_esperado = "\n".join(lineas) + "\n"

RESULTADOS.mkdir(parents=True, exist_ok=True)
ruta_resumen = RESULTADOS / "resumen_dataframe.txt"
ruta_resumen.write_text(resumen_esperado, encoding="utf-8", newline="\n")

ficha_tecnica = {
    "dataset_id": DATASET_ID,
    "enlace": ENLACE_DATASET,
    "fecha_consulta": FECHA_CONSULTA,
    "revision": REVISION,
    "configuracion": CONFIGURACION,
    "split": SPLIT,
    "muestra": False,
    "filas": filas,
    "columnas": columnas,
    "variables": [
        {"nombre": str(nombre), "tipo_pandas": str(df[nombre].dtype),
         "valores_nulos": int(df[nombre].isna().sum())}
        for nombre in df.columns
    ],
    "cobertura": cobertura,
    "registros_por_anio": {str(int(anio)): int(n) for anio, n in fechas.dt.year.value_counts().sort_index().items()},
    "registros_por_workingday": {str(int(categoria)): int(n) for categoria, n in df["workingday"].value_counts().sort_index().items()},
    "archivos_datos_en_hugging_face": archivos_datos,
    "memoria_dataframe_bytes": int(df.memory_usage(index=True, deep=True).sum()),
    "sha256_dataframe_y_esquema": huella,
    "sha256_resumen_utf8": hashlib.sha256(resumen_esperado.encode("utf-8")).hexdigest(),
    "python": platform.python_version(),
    "versiones": versiones,
}
ruta_ficha = RESULTADOS / "ficha_tecnica.json"
ruta_ficha.write_text(json.dumps(ficha_tecnica, ensure_ascii=False, indent=2) + "\n", encoding="utf-8", newline="\n")
print("Generados:")
print(ruta_resumen.relative_to(RAIZ).as_posix())
print(ruta_ficha.relative_to(RAIZ).as_posix())

Generados:


AE2/resultados/resumen_dataframe.txt
AE2/resultados/ficha_tecnica.json


## 6. Verificación del archivo generado

Se vuelve a leer el archivo en UTF-8 y se comprueba que coincide con el texto calculado desde `df`, incluidas sus dimensiones, columnas, tipos y nulos. Las comprobaciones fallan si hay una discrepancia; una ejecución correcta muestra el contenido completo del resumen y el mensaje de verificación.

In [9]:
resumen_leido = ruta_resumen.read_text(encoding="utf-8")
assert resumen_leido == resumen_esperado
assert f"Dataset: {DATASET_ID}" in resumen_leido
assert f"Revisión: {REVISION}" in resumen_leido
assert f"Filas: {df.shape[0]}" in resumen_leido
assert f"Columnas: {df.shape[1]}" in resumen_leido
for nombre in df.columns:
    assert f"{nombre} | {df[nombre].dtype} | {int(df[nombre].isna().sum())}" in resumen_leido

ficha_leida = json.loads(ruta_ficha.read_text(encoding="utf-8"))
assert ficha_leida["filas"] == len(df)
assert ficha_leida["columnas"] == len(df.columns)
assert ficha_leida["sha256_resumen_utf8"] == hashlib.sha256(ruta_resumen.read_bytes()).hexdigest()
assert df.shape == (17379, 17), "La revisión fijada debería mantener estas dimensiones."

print(resumen_leido)
print("Verificación correcta: el resumen UTF-8 corresponde al DataFrame revisado.")

RESUMEN DEL DATAFRAME — AE2
Dataset: t22000t/bike-sharing-tabular
Enlace: https://huggingface.co/datasets/t22000t/bike-sharing-tabular
Fecha de consulta: 2026-10-22
Revisión: 6b645e0cedbc5208fcca11ac94544b088478bf61
Configuración: default
Split: train
Muestra: no; split completo, sin filtros ni eliminación de registros.
Filas: 17379
Columnas: 17

NOMBRES, TIPOS DE DATOS Y VALORES NULOS
variable | tipo_pandas | valores_nulos
instant | int64 | 0
dteday | str | 0
season | int64 | 0
yr | int64 | 0
mnth | int64 | 0
hr | int64 | 0
holiday | int64 | 0
weekday | int64 | 0
workingday | int64 | 0
weathersit | int64 | 0
temp | float64 | 0
atemp | float64 | 0
hum | float64 | 0
windspeed | float64 | 0
casual | int64 | 0
registered | int64 | 0
cnt | int64 | 0

COBERTURA Y COHERENCIA BÁSICA
Periodo: 2011-01-01 a 2012-12-31
Fechas distintas: 731
Horas de calendario en el periodo: 17544
Horas observadas distintas: 17379
Horas sin registro: 165 (no imputadas; no equivalen a cero alquileres)
Duplicados f

## 7. Relación con la necesidad y diseño de la siguiente etapa

Los registros permiten agrupar `cnt` por `hr` y `workingday`, reconocer las horas disponibles y comparar los alquileres observados. En la siguiente etapa se prevé describir el promedio y la mediana de `cnt` para cada combinación hora–tipo de día, acompañados por el número de registros de cada grupo. Se revisará también la separación por año para contextualizar posibles diferencias de cobertura. Las comparaciones usarán horas registradas, sin rellenar ausencias con cero.

El proyecto describirá este sistema y este periodo históricos. Estos datos no permiten probar que trabajar o no trabajar cause un cambio en los alquileres, estimar usuarios únicos, conocer solicitudes no atendidas, evaluar estaciones concretas ni asegurar que los patrones se mantengan en otra ciudad o en la actualidad. Esta etapa no ejecuta esas comparaciones ni propone un modelo predictivo.

### Fuentes consultadas

- [Dataset en Hugging Face](https://huggingface.co/datasets/t22000t/bike-sharing-tabular), revisión fijada en el código.
- [Bike Sharing en UCI](https://archive.ics.uci.edu/dataset/275/bike+sharing), procedencia y diccionario original. DOI: [10.24432/C5W894](https://doi.org/10.24432/C5W894).
- [Hugging Face: carga de datasets](https://huggingface.co/docs/datasets/loading) y [uso con Pandas](https://huggingface.co/docs/datasets/use_with_pandas).
- [Pandas: guía de usuario](https://pandas.pydata.org/docs/user_guide/).

Fecha de consulta documental: 22 de octubre de 2026. Las definiciones de variables y las condiciones de uso deben leerse junto con la ficha y las referencias del documento Word.